# CIFAR-10 Generated Images Evaluation

This notebook evaluates diffusion model-generated CIFAR-10 images using three metrics:
1. **Accuracy**: Classification accuracy using pretrained CIFAR-10 ResNet20
2. **Logit Score**: Average log-probability of the true class (confidence measure)
3. **Inception Score (IS)**: Measures quality and diversity of generated images

## Evaluation Folders
- `cond_input`: Conditional generation with input embedding
- `cond_time`: Conditional generation with time embedding
- `cfg_eval_w1.0` to `cfg_eval_w12.0`: Classifier-free guidance with different weights
- `uncond`: Unconditional generation

## Imports and Setup

In [ ]:
# Standard library imports
import os
import re

# Third-party imports
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

# PyTorch and torchvision
from torchvision import transforms, datasets
from torchvision.models import inception_v3, Inception_V3_Weights

## Device Configuration

In [ ]:
# Select device (MPS for Apple Silicon, CUDA for NVIDIA, CPU as fallback)
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

## Download CIFAR-10 Test Set

**⚠️ IMPORTANT:** If you have already run this cell and the `cifar10_real/` folder exists with 10,000 images, you can **skip this cell** and proceed directly to **"Label Generation"**.

This cell downloads the CIFAR-10 test set and exports all 10,000 images to the `cifar10_real/` folder.

In [ ]:
real_folder = "cifar10_real"
os.makedirs(real_folder, exist_ok=True)

print("Loading CIFAR-10 test set...")
dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=None)

print(f"Start exporting {len(dataset)} CIFAR-10 test images to {real_folder}...")

for idx, (img_pil, label) in enumerate(tqdm(dataset, desc="Exporting images")):
    fname = f"{label}_{idx:05d}.png"  
    img_pil.save(os.path.join(real_folder, fname))

print("Export complete.")

## Label Generation

**⚠️ IMPORTANT:** If you have already run this cell and the label files (`labels_*.txt`) exist, you can **skip this cell** and proceed directly to **"Accuracy & Logit Score Evaluation"**.

Generate label files mapping image filenames to CIFAR-10 class IDs.

**CIFAR-10 Classes:**
- 0: airplane
- 1: automobile
- 2: bird
- 3: cat
- 4: deer
- 5: dog
- 6: frog
- 7: horse
- 8: ship
- 9: truck

In [ ]:
# CIFAR-10 class mapping
class_to_id = {
    "airplane": 0,
    "automobile": 1,
    "bird": 2,
    "cat": 3,
    "deer": 4,
    "dog": 5,
    "frog": 6,
    "horse": 7,
    "ship": 8,
    "truck": 9
}

def generate_label_file(folder, out_file):
    """
    Generate a label file mapping image filenames to CIFAR-10 class IDs.
    
    Args:
        folder: Directory containing images
        out_file: Output label file path
    """
    files = sorted(os.listdir(folder))
    pairs = []

    for f in files:
        # Skip non-image files
        if not (f.endswith(".png") or f.endswith(".jpg")):
            continue

        # Extract classname from filename
        match = re.match(r"([a-z]+)", f)
        if not match:
            print(f"Unrecognized class name: {f}")
            continue

        cls = match.group(1)  # e.g., "airplane"
        if cls not in class_to_id:
            print(f"Class not found: {cls}")
            continue

        label = class_to_id[cls]
        pairs.append(f"{f} {label}")

    # Write output
    with open(out_file, "w") as f:
        f.write("\n".join(pairs))

    print(f"Label file saved to: {out_file}")

In [ ]:
folders = [
    "cond_input",
    "cond_time",
    "cfg_eval_w1.0",
    "cfg_eval_w8",
    "cfg_eval_w3.0",
    "cfg_eval_w12.0",
    "cfg_eval_w5.0",
]

label_files = []

for f in folders:
    label_name = f"labels_{f}.txt"
    label_files.append(label_name)
    generate_label_file(f, label_name)

## Accuracy & Logit Score Evaluation

Uses pretrained CIFAR-10 ResNet20 classifier to evaluate:
- **Accuracy**: Percentage of correctly classified generated images
- **Logit Score**: Average log-probability of the true class (higher = better confidence)

In [ ]:
assert len(folders) == len(label_files), "Folders and label files must match!"

# Load pretrained CIFAR-10 classifier
print("Loading CIFAR-10 ResNet20 classifier...")
clf = torch.hub.load(
    "chenyaofo/pytorch-cifar-models",
    "cifar10_resnet20", 
    pretrained=True
).to(device)
clf.eval()
print("✔ Loaded pretrained CIFAR-10 classifier\n")

# CIFAR-10 normalization transform
transform_clf = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])


def evaluate_accuracy(folder, label_file, model):
    """
    Calculate classification accuracy.
    
    Args:
        folder: Image directory
        label_file: Label file path
        model: Classifier model
    
    Returns:
        Accuracy (fraction of correctly classified images)
    """
    total, correct = 0, 0
    with open(label_file) as f:
        pairs = [line.strip().split() for line in f.readlines()]
    
    for img_name, label in pairs:
        img_path = os.path.join(folder, img_name)
        img = Image.open(img_path).convert("RGB")
        x = transform_clf(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred = model(x).argmax(1).item()
        
        total += 1
        correct += (pred == int(label))
    
    return correct / total


def evaluate_logit_score(folder, label_file, model):
    """
    Calculate average logit score (log-probability of true class).
    
    Args:
        folder: Image directory
        label_file: Label file path
        model: Classifier model
    
    Returns:
        Average log-probability of the true class
    """
    scores = []
    with open(label_file) as f:
        pairs = [line.strip().split() for line in f.readlines()]
    
    for img_name, label in pairs:
        img_path = os.path.join(folder, img_name)
        img = Image.open(img_path).convert("RGB")
        x = transform_clf(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            logits = model(x)
            log_prob = F.log_softmax(logits, dim=1)
            scores.append(log_prob[0, int(label)].item())
    
    return sum(scores) / len(scores)


# Evaluate all folders
print("="*70)
print("ACCURACY & LOGIT SCORE EVALUATION")
print("="*70)

results = {}
for folder, label_file in zip(folders, label_files):
    print(f"\nEvaluating {folder}...")
    acc = evaluate_accuracy(folder, label_file, clf)
    logit = evaluate_logit_score(folder, label_file, clf)
    results[folder] = {"Accuracy": acc, "Logit": logit}
    print(f"  Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Logit Score: {logit:.4f}")


# Summary sorted by accuracy
print("\n" + "="*70)
print("SUMMARY - SORTED BY ACCURACY")
print("="*70)

folders_sorted_acc = sorted(results.items(), key=lambda x: x[1]["Accuracy"], reverse=True)
for i, (folder, vals) in enumerate(folders_sorted_acc, 1):
    medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{medal} {folder:20s}: Accuracy={vals['Accuracy']:.4f}, Logit={vals['Logit']:.4f}")

print("\n" + "="*70)
print("SUMMARY - SORTED BY LOGIT SCORE")
print("="*70)

folders_sorted_logit = sorted(results.items(), key=lambda x: x[1]["Logit"], reverse=True)
for i, (folder, vals) in enumerate(folders_sorted_logit, 1):
    medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{medal} {folder:20s}: Logit={vals['Logit']:.4f}, Accuracy={vals['Accuracy']:.4f}")

## Inception Score (IS) Evaluation

Measures quality and diversity of generated images using InceptionV3.

**Formula:** IS = exp(E[KL(p(y|x) || p(y))])

**Interpretation:**
- Higher IS = better quality and diversity
- CIFAR-10 typical range: 5-11
- IS ~ 1 indicates poor quality or no diversity

In [ ]:
# Configuration
batch_size = 16

folders_to_compare = [
    "cond_input",
    "cond_time",
    "uncond",
    "cfg_eval_w1.0",
    "cfg_eval_w8",
    "cfg_eval_w3.0",
    "cfg_eval_w12.0",
    "cfg_eval_w5.0",
]

# Load InceptionV3 with full classification head for IS
print("Loading InceptionV3 for Inception Score...")
weights = Inception_V3_Weights.IMAGENET1K_V1
inception = inception_v3(weights=weights, transform_input=False)  
inception.aux_logits = False
inception.eval().to(device)
print("✔ Loaded InceptionV3 for Inception Score calculation\n")


def load_and_normalize_image(img_path):
    """
    Load image and ensure it's in the correct range [0, 255] for PIL.
    Handles images that might be saved in different ranges (e.g., [-1, 1], [0, 1]).
    """
    img = Image.open(img_path).convert("RGB")
    img_array = np.array(img)
    
    # Check the value range and normalize if needed
    min_val, max_val = img_array.min(), img_array.max()
    
    # If image appears to be in [-1, 1] or [0, 1] range
    if max_val <= 1.0 and min_val >= -1.0:
        # Rescale to [0, 255]
        img_array = ((img_array - min_val) / (max_val - min_val) * 255).astype(np.uint8)
        img = Image.fromarray(img_array)
    elif max_val <= 255 and min_val >= 0:
        # Already in correct range
        pass
    else:
        print(f"Warning: Unexpected value range [{min_val}, {max_val}] in {img_path}")
    
    return img


# Standard InceptionV3 preprocessing
transform = transforms.Compose([
    transforms.Resize((299, 299), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),  # Converts PIL [0,255] to tensor [0,1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def compute_inception_score(folder, batch_size=16):
    """
    Compute Inception Score for a folder of images.
    
    Args:
        folder: Directory containing images
        batch_size: Batch size for processing
    
    Returns:
        IS score (scalar)
    """
    files = [f for f in os.listdir(folder) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    if len(files) == 0:
        print(f"[WARN] No images in {folder}")
        return None

    print(f"  Computing IS for {len(files)} images...")
    
    probs = []
    for i in tqdm(range(0, len(files), batch_size), desc=f"Processing {folder}"):
        batch_files = files[i:i+batch_size]
        imgs = []
        for fname in batch_files:
            try:
                img = load_and_normalize_image(os.path.join(folder, fname))
                imgs.append(transform(img))
            except Exception as e:
                print(f"  Warning: Skipping {fname}: {e}")
                continue
        if len(imgs) == 0:
            continue

        x = torch.stack(imgs).to(device)
        with torch.no_grad():
            logits = inception(x)
            # Get probability distribution p(y|x)
            p_yx = F.softmax(logits, dim=1).cpu().numpy()
            probs.append(p_yx)

    if len(probs) == 0:
        return None

    probs = np.concatenate(probs, axis=0)  # Shape: (N, 1000)
    
    # Compute marginal distribution p(y)
    p_y = np.mean(probs, axis=0, keepdims=True)  # Shape: (1, 1000)
    
    # Compute KL divergence: KL(p(y|x) || p(y))
    kl = probs * (np.log(probs + 1e-10) - np.log(p_y + 1e-10))
    kl_per_sample = np.sum(kl, axis=1)  # Sum over classes
    
    # Mean IS
    is_score = float(np.exp(np.mean(kl_per_sample)))
    
    return is_score


# Compute IS for all folders
print("="*70)
print("INCEPTION SCORE EVALUATION")
print("="*70)
print()

scores = {}
for folder in folders_to_compare:
    print(f"\nProcessing: {folder}")
    is_score = compute_inception_score(folder, batch_size=batch_size)
    if is_score is not None:
        scores[folder] = is_score
        print(f"  ✓ IS = {is_score:.4f}")
    else:
        scores[folder] = None
        print(f"  ❌ Calculation failed")


# Summary sorted by IS (higher is better)
print("\n" + "="*70)
print("INCEPTION SCORE SUMMARY (higher is better)")
print("="*70)

sorted_scores = sorted(
    [(k, v) for k, v in scores.items() if v is not None],
    key=lambda x: x[1],
    reverse=True
)

for i, (folder, score) in enumerate(sorted_scores, 1):
    medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{medal} {folder:20s}: IS = {score:.4f}")

for folder, score in scores.items():
    if score is None:
        print(f"❌ {folder:20s}: Calculation failed")

print("\n" + "="*70)
print("INTERPRETATION GUIDE")
print("="*70)
print("IS 10-11 : Excellent quality for CIFAR-10 conditional generation")
print("IS 8-10  : Good quality")
print("IS 5-8   : Moderate quality, typical for unconditional models")
print("IS < 5   : Poor quality or lack of diversity")
print("IS ~ 1   : Preprocessing error or mode collapse")
print("="*70)